In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import sklearn
from scipy.signal import detrend
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.manifold import TSNE
# import os, json, uuid, hashlib, platform, inspect
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value.to_string() if isinstance(value, pd.DataFrame) else value)

RAW_DATA_ROOT = Path(os.environ.get("PINPAD_DATA_ROOT", "/home/cse/pinpad/myenv/GNURadio_1/Correlation_Responses/OneSubcarrierLowDataRate (Copy)")).expanduser()
OUTPUT_ROOT = Path(os.environ.get("PINPAD_RF_OUTPUT", str(Path.cwd() / "RF_Results"))).expanduser()
RUN_TRAINING = False
RUN_INFERENCE = False
RUN_SAVED_PLOTS = False
RUN_SEEDS = [42, 43, 44, 45, 46]
RESPONSE_LENGTH = 3624
N_PER_BUTTON = 200
CLASS_ORDER = [f"B{i}" for i in range(12)]
DOMAINS = ["CC", "CR", "CCR"]
PCA_COMPONENTS = 64
CORAL_EPSILON = 1e-5
N_TREES = 300
EVALUATION_SPLITS_CSV = None
SAVED_MODEL_PATH = None
INPUT_PATH = None
LABELS_PATH = None
INPUT_DOMAIN = "CC"
ARCHIVE_RUN_DIR = None
TSNE_MAX_POINTS = 3000
TSNE_SEED = 42
PREPROCESSING = {"representation":"magnitude", "drift_removal":"linear", "normalization":"per_response_zscore", "std_epsilon":1e-12, "signal_length":RESPONSE_LENGTH}
SOURCE = [(c, "R1") for c in range(1, 6)]
ADAPT = {"CC":[(c, "R1") for c in range(6, 11)], "CR":[(c, "R2") for c in range(1, 6)], "CCR":[(c, "R2") for c in range(11, 16)]}
TEST = {"CC":[(c, "R1") for c in range(16, 26)], "CR":[(c, "R3") for c in range(1, 6)], "CCR":[(c, "R3") for c in range(26, 36)]}
plt.rcParams.update({"font.family":"sans-serif", "font.size":9, "pdf.fonttype":42, "ps.fonttype":42})


In [ ]:
%pip install adapt==0.4.5

  Using cached scikeras-0.13.0-py3-none-any.whl.metadata (3.1 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.3/21.3 MB 3.1 MB/s  0:00:06a 0:00:01m eta 0:00:01
Using cached scikeras-0.13.0-py3-none-any.whl (26 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [adapt]

[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
def new_directory(parent, prefix):
    path = Path(parent) / (prefix + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ_") + uuid.uuid4().hex[:8])
    path.mkdir(parents=True, exist_ok=False)
    return path

def write_json(path, data):
    Path(path).write_text(json.dumps(data, indent=2, allow_nan=False))

def resolve_collections():
    expected = set(SOURCE + [k for v in ADAPT.values() for k in v] + [k for v in TEST.values() for k in v])
    mapping = {}
    manifest = RAW_DATA_ROOT / "collection_paths.csv"
    if manifest.is_file():
        frame = pd.read_csv(manifest)
        if not {"card", "reader", "directory"}.issubset(frame.columns):
            raise ValueError("collection_paths.csv requires card, reader, directory")
        for row in frame.itertuples(index=False):
            key = (int(row.card), str(row.reader).upper())
            if key in mapping:
                raise ValueError(f"Duplicate collection: {key}")
            p = Path(str(row.directory)).expanduser()
            mapping[key] = p if p.is_absolute() else RAW_DATA_ROOT / p
    else:
        for card, reader in sorted(expected):
            role = "Source" if (card, reader) in SOURCE else next(d for d in DOMAINS if (card, reader) in ADAPT[d] + TEST[d])
            candidates = [RAW_DATA_ROOT / reader / f"card{card}", RAW_DATA_ROOT / reader.lower() / f"card{card}", RAW_DATA_ROOT / f"card{card}" / reader, RAW_DATA_ROOT / role / reader / f"card{card}", RAW_DATA_ROOT / role / reader.lower() / f"card{card}"]
            if role == "Source":
                candidates.append(RAW_DATA_ROOT / "Source" / f"card{card}")
            if role == "CC":
                candidates.append(RAW_DATA_ROOT / "CC" / f"card{card}")
            hits = sorted({p.resolve() for p in candidates if p.is_dir()})
            if len(hits) != 1:
                raise FileNotFoundError(f"Expected one directory for card {card}, {reader}; found {hits}. Set collection_paths.csv.")
            mapping[(card, reader)] = hits[0]
    missing = [k for k in sorted(expected) if k not in mapping or not mapping[k].is_dir()]
    if missing:
        raise FileNotFoundError(f"Missing collections: {missing}")
    selected = {k:mapping[k].resolve() for k in expected}
    if len(set(selected.values())) != len(selected):
        raise ValueError("Different card-reader collections resolve to the same directory")
    return selected

def read_collection(key, mapping):
    parts, labels, ids = [], [], []
    for button in range(len(CLASS_ORDER)):
        hits = [p for p in (mapping[key] / f"button{button}.npy", mapping[key] / f"Button{button}.npy") if p.is_file()]
        if len(hits) != 1:
            raise FileNotFoundError(f"Expected one button{button}.npy in {mapping[key]}")
        x = np.load(hits[0], allow_pickle=False)
        if x.shape != (N_PER_BUTTON, RESPONSE_LENGTH) or not np.issubdtype(x.dtype, np.number) or not np.isfinite(x).all():
            raise ValueError(f"Invalid response array: {hits[0]} {x.shape} {x.dtype}")
        parts.append(x)
        labels.extend([button] * len(x))
        ids.extend([f"{key[1]}/card{key[0]}/button{button}/row{i}" for i in range(len(x))])
    return np.concatenate(parts), np.asarray(labels, dtype=np.int64), np.asarray(ids)

def preprocess(signals, config):
    x = np.asarray(signals)
    if x.ndim != 2 or x.shape[1] != config["signal_length"] or not np.isfinite(x).all():
        raise ValueError("Expected finite response rows with the configured signal length")
    if (config["representation"], config["drift_removal"], config["normalization"]) != ("magnitude", "linear", "per_response_zscore"):
        raise ValueError("Unsupported preprocessing")
    x = detrend(np.abs(x).astype(np.float64), axis=1, type="linear")
    std = x.std(axis=1, keepdims=True)
    if np.any(std <= config["std_epsilon"]):
        raise ValueError("Response has negligible variation after drift removal")
    return ((x-x.mean(axis=1, keepdims=True))/std).astype(np.float32)

def read_group(keys, mapping):
    records = [read_collection(k, mapping) for k in keys]
    return {"x":np.concatenate([preprocess(t[0], PREPROCESSING) for t in records]), "y":np.concatenate([t[1] for t in records]), "ids":np.concatenate([t[2] for t in records]), "cards":np.concatenate([np.full(len(t[1]), k[0]) for k,t in zip(keys,records)]), "readers":np.concatenate([np.full(len(t[1]), k[1]) for k,t in zip(keys,records)])}


In [ ]:
def build_evaluation_splits():
    if len(RUN_SEEDS) < 1 or len(set(RUN_SEEDS)) != len(RUN_SEEDS):
        raise ValueError("RUN_SEEDS must be nonempty and unique")
    if EVALUATION_SPLITS_CSV is None:
        return [{d:list(TEST[d]) for d in DOMAINS} for _ in RUN_SEEDS], "repeated_seeds_fixed_evaluation_collections"
    frame = pd.read_csv(EVALUATION_SPLITS_CSV)
    if not {"run", "domain", "card", "reader"}.issubset(frame.columns):
        raise ValueError("Evaluation split CSV requires run, domain, card, reader")
    if set(frame["run"]) != set(range(1, len(RUN_SEEDS)+1)) or not set(frame.domain).issubset(DOMAINS):
        raise ValueError("Evaluation runs or domains do not match the configuration")
    splits = []
    for i in range(1, len(RUN_SEEDS)+1):
        one = {}
        for domain in DOMAINS:
            rows = frame[(frame["run"] == i) & (frame.domain == domain)]
            keys = [(int(r.card), str(r.reader).upper()) for r in rows.itertuples(index=False)]
            if not keys or len(set(keys)) != len(keys) or not set(keys).issubset(TEST[domain]):
                raise ValueError(f"Invalid evaluation collections: run {i}, {domain}")
            one[domain] = keys
        splits.append(one)
    return splits, "explicit_card_collection_manifest"

def audit_inputs(mapping, output):
    hashes, records = {}, []
    roles = {"source":SOURCE, **{f"adapt_{d}":ADAPT[d] for d in DOMAINS}, **{f"test_{d}":TEST[d] for d in DOMAINS}}
    for role, keys in roles.items():
        for key in keys:
            raw, y, ids = read_collection(key, mapping)
            features = preprocess(raw, PREPROCESSING)
            for signal, sample_id in zip(features, ids):
                h = hashlib.sha256(np.ascontiguousarray(signal).view(np.uint8)).hexdigest()
                records.append({"role":role, "sample_id":sample_id, "processed_sha256":h})
                hashes.setdefault(h, set()).add(role)
    frame = pd.DataFrame(records)
    frame.to_csv(output / "input_inventory.csv", index=False)
    invalid = {h for h,roles in hashes.items() if any(r.startswith("test_") for r in roles) and any(not r.startswith("test_") for r in roles)}
    if invalid:
        frame[frame.processed_sha256.isin(invalid)].to_csv(output / "overlapping_inputs.csv", index=False)
        raise ValueError("Training/adaptation and evaluation contain identical processed responses")


In [ ]:
def common_target_reference(target_features):
    arrays = [np.asarray(target_features[d], dtype=np.float64) for d in DOMAINS]
    if any(x.ndim != 2 or len(x) < 2 for x in arrays):
        raise ValueError("Each target domain requires at least two feature rows")
    means = [x.mean(axis=0) for x in arrays]
    covariance = np.mean([np.cov(x, rowvar=False, ddof=1) for x in arrays], axis=0)
    total = sum(len(x) for x in arrays)
    reference = np.concatenate([(x-mu)*np.sqrt((total-1)/(len(arrays)*(len(x)-1))) for x,mu in zip(arrays,means)])
    np.testing.assert_allclose(reference.mean(axis=0), 0, atol=1e-10)
    np.testing.assert_allclose(np.cov(reference, rowvar=False, ddof=1), covariance, rtol=1e-8, atol=1e-10)
    return reference, np.mean(means, axis=0), covariance

def fit_coral(source_features, target_features, seed):
    from adapt.feature_based import CORAL
    source = np.asarray(source_features, dtype=np.float64)
    reference, target_mean, target_covariance = common_target_reference(target_features)
    source_mean = source.mean(axis=0)
    adapter = CORAL(lambda_=CORAL_EPSILON, verbose=0, random_state=seed)
    aligned = adapter.fit_transform(source-source_mean, reference) + target_mean
    transform = adapter.transform(np.eye(source.shape[1]), domain="src")
    np.testing.assert_allclose(adapter.Cs_, np.cov(source, rowvar=False, ddof=1)+CORAL_EPSILON*np.eye(source.shape[1]), rtol=1e-7, atol=1e-9)
    np.testing.assert_allclose(adapter.Ct_, target_covariance+CORAL_EPSILON*np.eye(source.shape[1]), rtol=1e-7, atol=1e-9)
    np.testing.assert_allclose(aligned, (source-source_mean)@transform+target_mean, rtol=1e-7, atol=1e-8)
    return aligned, {"source_mean":source_mean, "target_mean":target_mean, "transform":transform, "source_covariance_regularized":adapter.Cs_, "target_covariance_regularized":adapter.Ct_}

def make_forest(seed):
    return RandomForestClassifier(n_estimators=N_TREES, criterion="gini", max_features="sqrt", min_samples_split=2, min_samples_leaf=1, max_depth=None, bootstrap=True, class_weight=None, n_jobs=-1, random_state=seed)

def feature_view(bundle, signals, model_kind="adapted", domain="CC"):
    if model_kind not in ("source", "adapted") or domain not in ["Source"]+DOMAINS:
        raise ValueError("Invalid model or domain")
    z = bundle["pca"].transform(preprocess(signals, bundle["preprocessing"]))
    if model_kind == "adapted" and domain == "Source":
        z = (z-bundle["source_mean"])@bundle["transform"]+bundle["target_mean"]
    return z

def predict_saved(bundle, signals, model_kind="adapted", domain="CC"):
    z = feature_view(bundle, signals, model_kind, domain)
    forest = bundle["rf"] if model_kind == "adapted" else bundle["rf_source"]
    probabilities = forest.predict_proba(z)
    predicted = forest.classes_[np.argmax(probabilities, axis=1)]
    return predicted, probabilities, z


In [ ]:
def save_predictions(path, labels, predicted, probabilities, features, sample_ids, cards, readers, domain, model_kind):
    counts = confusion_matrix(labels, predicted, labels=np.arange(len(CLASS_ORDER)))
    normalized = np.divide(counts, counts.sum(axis=1, keepdims=True), out=np.zeros_like(counts, dtype=float), where=counts.sum(axis=1, keepdims=True)>0)
    np.savez_compressed(path, true_labels=labels, predicted_labels=predicted, probabilities=probabilities, features=features, sample_ids=sample_ids, cards=cards, readers=readers, domain=np.asarray(domain), model_kind=np.asarray(model_kind), class_order=np.asarray(CLASS_ORDER), confusion_counts=counts, confusion_row_normalized=normalized)
    return {"domain":domain, "model":model_kind, "n":len(labels), "correct":int(np.sum(labels==predicted)), "accuracy_percent":float(100*accuracy_score(labels,predicted))}

def train_all():
    import importlib.metadata
    if importlib.metadata.version("adapt") != "0.4.5":
        raise RuntimeError("Install adapt==0.4.5 for this training configuration")
    mapping = resolve_collections()
    evaluation_splits, protocol = build_evaluation_splits()
    run_dir = new_directory(OUTPUT_ROOT, "rf_")
    write_json(run_dir / "config.json", {"seeds":RUN_SEEDS, "evaluation_protocol":protocol, "evaluation_splits":evaluation_splits, "source":SOURCE, "adaptation":ADAPT, "preprocessing":PREPROCESSING, "pca_components":PCA_COMPONENTS, "coral_epsilon":CORAL_EPSILON, "trees":N_TREES, "python":platform.python_version(), "numpy":np.__version__, "sklearn":sklearn.__version__, "adapt":importlib.metadata.version("adapt")})
    audit_inputs(mapping, run_dir)
    source = read_group(SOURCE, mapping)
    targets = {d:read_group(ADAPT[d], mapping) for d in DOMAINS}
    rows = []
    for index, seed in enumerate(RUN_SEEDS):
        out = run_dir / f"run_{index+1:02d}"
        out.mkdir(exist_ok=False)
        print(f"Training run {index+1}/{len(RUN_SEEDS)}, seed {seed}", flush=True)
        pca = PCA(n_components=PCA_COMPONENTS, whiten=False, svd_solver="full", random_state=seed)
        source_z = pca.fit_transform(source["x"])
        target_z = {d:pca.transform(targets[d]["x"]) for d in DOMAINS}
        aligned, alignment = fit_coral(source_z, target_z, seed)
        baseline = make_forest(seed).fit(source_z, source["y"])
        adapted = make_forest(seed).fit(aligned, source["y"])
        for forest in (baseline, adapted):
            np.testing.assert_array_equal(forest.classes_, np.arange(len(CLASS_ORDER)))
        bundle = {"pca":pca, "rf_source":baseline, "rf":adapted, **alignment, "preprocessing":PREPROCESSING, "classes":CLASS_ORDER, "seed":seed, "evaluation_collections":evaluation_splits[index]}
        joblib.dump(bundle, out / "RF_models.joblib", compress=3)
        np.savez_compressed(out / "CORAL_Parameters.npz", **alignment, explained_variance_ratio=pca.explained_variance_ratio_)
        for domain in DOMAINS:
            for key in evaluation_splits[index][domain]:
                raw, labels, ids = read_collection(key, mapping)
                for model_kind in ("source", "adapted"):
                    predicted, probabilities, features = predict_saved(bundle, raw, model_kind, domain)
                    metric = save_predictions(out / f"{model_kind}_{domain}_{key[1]}_card{key[0]}.npz", labels, predicted, probabilities, features, ids, np.full(len(labels), key[0]), np.full(len(labels), key[1]), domain, model_kind)
                    rows.append({"run":index+1, "seed":seed, "card":key[0], "reader":key[1], **metric})
        pd.DataFrame(rows).to_csv(run_dir / "metrics_per_collection.csv", index=False)
    return run_dir, pd.DataFrame(rows)


In [ ]:
def summarize_metrics(frame, output):
    records = []
    for (run, model), group in frame.groupby(["run", "model"], sort=True):
        accuracies = []
        for domain in DOMAINS:
            one = group[group.domain==domain]
            if one.empty:
                raise ValueError(f"No metrics for run {run}, {model}, {domain}")
            accuracy = 100*one.correct.sum()/one.n.sum()
            accuracies.append(accuracy)
            records.append({"run":run, "model":model, "domain":domain, "accuracy_percent":accuracy})
        records.extend([{"run":run,"model":model,"domain":"target_combined_sample_weighted","accuracy_percent":100*group.correct.sum()/group.n.sum()}, {"run":run,"model":model,"domain":"target_macro_domain_mean","accuracy_percent":float(np.mean(accuracies))}])
    per_run = pd.DataFrame(records)
    summary = per_run.groupby(["model","domain"]).accuracy_percent.agg(["mean","std","count"]).rename(columns={"mean":"accuracy_percent_mean","std":"sample_sd_pp","count":"n_runs"}).reset_index()
    per_run.to_csv(output / "metrics_per_run.csv", index=False)
    summary.to_csv(output / "metrics_summary.csv", index=False)
    np.savez_compressed(output / "RF_Recognition_Performance.npz", models=summary.model.to_numpy(dtype=str), domains=summary.domain.to_numpy(dtype=str), mean_accuracy_percent=summary.accuracy_percent_mean.to_numpy(), sample_sd_pp=summary.sample_sd_pp.to_numpy(), n_runs=summary.n_runs.to_numpy(), per_run_accuracy_percent=per_run.accuracy_percent.to_numpy(), per_run_number=per_run.run.to_numpy(), per_run_models=per_run.model.to_numpy(dtype=str), per_run_domains=per_run.domain.to_numpy(dtype=str))
    display(per_run)
    display(summary)
    return summary

if RUN_TRAINING:
    TRAINING_RUN_DIR, COLLECTION_METRICS = train_all()
    SUMMARY = summarize_metrics(COLLECTION_METRICS, TRAINING_RUN_DIR)
    print("Results:", TRAINING_RUN_DIR)


In [ ]:
def plot_confusion(counts, title, output_stem):
    normalized = np.divide(counts, counts.sum(axis=1, keepdims=True), out=np.zeros_like(counts, dtype=float), where=counts.sum(axis=1, keepdims=True)>0)
    fig, ax = plt.subplots(figsize=(6,5))
    im = ax.imshow(normalized, vmin=0, vmax=1, cmap="Blues")
    ax.set(xticks=np.arange(len(CLASS_ORDER)), yticks=np.arange(len(CLASS_ORDER)), xticklabels=CLASS_ORDER, yticklabels=CLASS_ORDER, xlabel="Predicted button", ylabel="True button", title=title)
    for i in range(len(CLASS_ORDER)):
        for j in range(len(CLASS_ORDER)):
            ax.text(j,i,f"{normalized[i,j]:.2f}",ha="center",va="center",fontsize=6,color="white" if normalized[i,j]>.5 else "black")
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    fig.savefig(str(output_stem)+".png", dpi=300)
    fig.savefig(str(output_stem)+".pdf")
    plt.show()
    plt.close(fig)
    display(pd.DataFrame(counts,index=CLASS_ORDER,columns=CLASS_ORDER))


def replay_evaluation(run_dir):
    output = new_directory(Path(run_dir) / "plots", "plots_")
    metrics = pd.read_csv(Path(run_dir) / "metrics_per_collection.csv")
    summary = summarize_metrics(metrics, output)
    for domain in DOMAINS:
        counts = np.zeros((len(CLASS_ORDER),len(CLASS_ORDER)), dtype=np.int64)
        for path in sorted(Path(run_dir).glob(f"run_*/adapted_{domain}_*.npz")):
            with np.load(path, allow_pickle=False) as z:
                counts += z["confusion_counts"]
        if counts.sum()==0:
            raise FileNotFoundError(f"No adapted predictions for {domain}")
        np.savez_compressed(output / f"RF_Confusion_{domain}.npz", counts=counts, row_normalized=counts/counts.sum(axis=1,keepdims=True), class_order=np.asarray(CLASS_ORDER), aggregation=np.asarray("pooled_evaluation_rows_across_runs"))
        plot_confusion(counts, domain, output / f"RF_Confusion_{domain}")
    fig, ax = plt.subplots(figsize=(5,3))
    x = np.arange(len(DOMAINS))
    for j, model in enumerate(("source","adapted")):
        rows = summary[(summary.model==model)&summary.domain.isin(DOMAINS)].set_index("domain").loc[DOMAINS]
        errors = rows.sample_sd_pp.to_numpy() if (rows.n_runs>1).all() else None
        ax.bar(x+(j-.5)*.35, rows.accuracy_percent_mean, .35, yerr=errors, capsize=3, label=model)
    ax.set(xticks=x,xticklabels=DOMAINS,ylabel="Recognition accuracy (%)",ylim=(0,100))
    ax.legend()
    fig.tight_layout()
    fig.savefig(output / "RF_Recognition.png",dpi=300)
    fig.savefig(output / "RF_Recognition.pdf")
    plt.show()
    plt.close(fig)
    return output

if RUN_TRAINING:
    PLOT_OUTPUT = replay_evaluation(TRAINING_RUN_DIR)
elif RUN_SAVED_PLOTS:
    if ARCHIVE_RUN_DIR is None:
        raise ValueError("Set ARCHIVE_RUN_DIR to a completed RF run")
    PLOT_OUTPUT = replay_evaluation(Path(ARCHIVE_RUN_DIR))


In [ ]:
def make_tsne(run_dir, run_number=1):
    run_folder = Path(run_dir) / f"run_{run_number:02d}"
    rng = np.random.default_rng(TSNE_SEED)
    features, labels, domains, ids = [], [], [], []
    for domain in DOMAINS:
        domain_features, domain_labels, domain_ids = [], [], []
        for path in sorted(run_folder.glob(f"adapted_{domain}_*.npz")):
            with np.load(path, allow_pickle=False) as z:
                domain_features.append(z["features"])
                domain_labels.append(z["true_labels"])
                domain_ids.append(z["sample_ids"])
        if not domain_features:
            raise FileNotFoundError(f"Missing saved features for {domain}")
        x,y,sample_ids = np.concatenate(domain_features),np.concatenate(domain_labels),np.concatenate(domain_ids)
        selected = np.sort(rng.choice(len(x),min(len(x),max(1,TSNE_MAX_POINTS//len(DOMAINS))),replace=False))
        features.append(x[selected]); labels.append(y[selected]); ids.append(sample_ids[selected]); domains.extend([domain]*len(selected))
    x,y,sample_ids = np.concatenate(features),np.concatenate(labels),np.concatenate(ids)
    if len(x)<4:
        raise ValueError("Too few t-SNE points")
    params = {"n_components":2, "perplexity":min(30.,(len(x)-1)/3), "init":"pca", "learning_rate":"auto", "random_state":TSNE_SEED}
    params["max_iter" if "max_iter" in inspect.signature(TSNE).parameters else "n_iter"] = 1000
    xy = TSNE(**params).fit_transform(x)
    output = new_directory(Path(run_dir) / "plots", "tsne_")
    path = output / "RF_TSNE_After_Adaptation.npz"
    np.savez_compressed(path, coordinates=xy, features=x, true_labels=y, domains=np.asarray(domains), sample_ids=sample_ids, class_order=np.asarray(CLASS_ORDER), parameters_json=np.asarray(json.dumps(params)), run_number=np.asarray(run_number))
    plot_saved_tsne(path, output)
    return path

def plot_saved_tsne(path, output):
    with np.load(path, allow_pickle=False) as z:
        xy,y,domains = z["coordinates"],z["true_labels"],z["domains"]
    fig, axes = plt.subplots(1,2,figsize=(10,4))
    for j,name in enumerate(CLASS_ORDER):
        mask = y==j
        axes[0].scatter(xy[mask,0],xy[mask,1],s=7,alpha=.65,label=name)
    colors = {"CC":"#b22222","CR":"#1b7a3a","CCR":"#1646a0"}
    for domain in DOMAINS:
        mask = domains==domain
        axes[1].scatter(xy[mask,0],xy[mask,1],s=7,alpha=.65,color=colors[domain],label=domain)
    for ax in axes:
        ax.set(xlabel="t-SNE 1",ylabel="t-SNE 2")
        ax.legend(fontsize=6,ncol=3)
    fig.tight_layout()
    fig.savefig(Path(output) / "RF_TSNE_After_Adaptation.png",dpi=300)
    fig.savefig(Path(output) / "RF_TSNE_After_Adaptation.pdf")
    plt.show()
    plt.close(fig)

if RUN_TRAINING:
    TSNE_PATH = make_tsne(TRAINING_RUN_DIR)


In [ ]:
# def run_saved_inference():
#     if SAVED_MODEL_PATH is None or INPUT_PATH is None:
#         raise ValueError("Set SAVED_MODEL_PATH to RF_models.joblib and INPUT_PATH to a response .npy file")
#     bundle = joblib.load(Path(SAVED_MODEL_PATH))
#     if bundle["classes"] != CLASS_ORDER:
#         raise ValueError("Model class order differs")
#     raw = np.load(Path(INPUT_PATH), allow_pickle=False)
#     predicted, probabilities, features = predict_saved(bundle, raw, "adapted", INPUT_DOMAIN)
#     output = new_directory(OUTPUT_ROOT, "rf_inference_")
#     frame = pd.DataFrame({"row":np.arange(len(raw)),"predicted_button":np.asarray(CLASS_ORDER)[predicted],"confidence":probabilities.max(axis=1)})
#     payload = {"predicted_labels":predicted,"probabilities":probabilities,"features":features,"class_order":np.asarray(CLASS_ORDER),"domain":np.asarray(INPUT_DOMAIN)}
#     if LABELS_PATH is not None:
#         labels = np.load(Path(LABELS_PATH),allow_pickle=False)
#         if labels.shape != predicted.shape or not np.issubdtype(labels.dtype,np.integer) or not np.isin(labels,np.arange(len(CLASS_ORDER))).all():
#             raise ValueError("Labels must be integer class indices aligned with response rows")
#         payload["true_labels"] = labels
#         counts = confusion_matrix(labels,predicted,labels=np.arange(len(CLASS_ORDER)))
#         payload["confusion_counts"] = counts
#         print(f"Accuracy: {100*accuracy_score(labels,predicted):.6f}%")
#         plot_confusion(counts,INPUT_DOMAIN,output / "RF_Inference_Confusion")
#     np.savez_compressed(output / "RF_Inference.npz",**payload)
#     frame.to_csv(output / "RF_Inference.csv",index=False)
#     display(frame)
#     print("Saved:",output)
#     return output

# if RUN_INFERENCE:
    # INFERENCE_OUTPUT = run_saved_inference()
